In [ ]:
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """
    Get the conversion factor between two currencies.

    """
    uri = f"https://v6.exchangerate-api.com/v6/{os.getenv('YOUR_API_KEY')}/pair/{base_currency}/{target_currency}"
    response = requests.get(uri)
    data = response.json()
    if data["result"] == "success":
        return data["conversion_rate"]
    else:
        raise ValueError(f"Error fetching conversion factor: {data['error-type']}") 

@tool
def convert(base_currency_value:int,conversion_rate:Annotated[float, InjectedToolArg("conversion_rate")])-> float:
    """
    Convert a value from the base currency to the target currency using the conversion rate.
   
    """
    return base_currency_value * conversion_rate


In [ ]:
get_conversion_factor.invoke({"base_currency": "USD", "target_currency": "NPR"})


In [ ]:
convert.invoke({"base_currency_value": 100, "conversion_rate": 132.5})


In [ ]:
llm=ChatOpenAI(model="gpt-3.5-turbo", temperature=0.9)

In [ ]:
llmWithTools=llm.bind_tools([get_conversion_factor,convert])

In [ ]:
messages=[HumanMessage(content="What is the conversion factor between USD and NPR? and based on that what is the value of 58 USD in NPR?")]


In [ ]:


aiMessages=llmWithTools.invoke(messages)
messages.append(aiMessages)

In [ ]:
import json

for tool_call in aiMessages.tool_calls:
    if tool_call.name=="get_conversion_factor":
        toolMessage1=get_conversion_factor.invoke(tool_call)
        print(toolMessage1)
        conversion_rate=json.loads(toolMessage1.content)["conversion_rate"]
        messages.append(toolMessage1)

    if tool_call.name=="convert":
        tool_call.args["conversion_rate"]=conversion_rate
        toolMessage2=convert.invoke(tool_call)
        messages.append(toolMessage2)

   


In [ ]:
llmWithTools.invoke(messages).content